In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from PIL import Image
from io import BytesIO
import io
import ast
import matplotlib.pyplot as plt
import base64
import json
import os
import re
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import torch
import gc
gc.collect() 
torch.cuda.empty_cache() 
torch.cuda.ipc_collect()

/home/nicolas/.conda/envs/vqa_rad/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_json("/home/nicolas/VQA-RAD-Matiaz/Proj_Mosca/outputs/Qwen_Qwen3.5-9B_Sumarization.jsonl", lines=True)
df

,id,model,sumarization
0,0,Qwen/Qwen3.5-9B,"{\n ""hosts"": [\n {\n ""sex"": ""F"",\n ..."
1,1,Qwen/Qwen3.5-9B,"{\n ""hosts"": [\n {\n ""sex"": ""M"",\n ..."
2,2,Qwen/Qwen3.5-9B,"{\n ""hosts"": [\n {\n ""sex"": ""M"",\n ..."
3,3,Qwen/Qwen3.5-9B,"{\n ""hosts"": [\n {\n ""sex"": ""F"",\n ..."
4,4,Qwen/Qwen3.5-9B,"{\n ""hosts"": [\n {\n ""sex"": ""M"",\n ..."
...,...,...,...
98,100,Qwen/Qwen3.5-9B,"{\n ""hosts"": [\n {\n ""sex"": ""F"",\n ..."
99,101,Qwen/Qwen3.5-9B,"{\n ""hosts"": [\n {\n ""sex"": ""F"",\n ..."
100,102,Qwen/Qwen3.5-9B,"{\n ""hosts"": [\n {\n ""sex"": null,\n ..."
101,103,Qwen/Qwen3.5-9B,"{\n ""hosts"": [\n {\n ""sex"": null,\n ..."


In [3]:
def load_model(model_name):
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4"
    )
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=quant_config,
        device_map="auto",
        low_cpu_mem_usage=True,
        offload_buffers=True
    )
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.padding_side = "left"
    return model, tokenizer

In [4]:
def build_messages(texto):
    return [
        {
        "role": "system",
        "content": [{"type": "text", "text": (
            
            "You are an assistant specialized in validating abstracts of scientific texts."
            """Following the example of the following JSON object structure, you must check if the information entered is in the correct fields and if it is not, you must return a new JSON object with the information correctly arranged. In cases where the information is already correctly arranged, simply return a comma.
{
  "hosts": [
    {
      "sex": "M",
      "age": 35,
      "species": "Homo sapiens",
      "type": "Human"
    },
    {
      "sex": "F",
      "age": 4,
      "species": "Canis lupus familiaris",
      "type": "CompanionAnimal"
    },
    {
      "sex": "M",
      "age": 2,
      "species": "Bos taurus",
      "type": "LivestockAnimal"
    },
    {
      "sex": "F",
      "age": 1,
      "species": "Didelphis marsupialis",
      "type": "WildAnimal"
    }
  ],
  "vectors": [
    {
      "species": "Dermatobia hominis",
      "myiasis_type": "Primary",
      "synonyms": ["berne", "botfly"]
    }
    {
      "species": "Dermatobia hominis",
      "myiasis_type": "Secondary",
      "synonyms": ["berne", "botfly"]
    }
  ],
  "locations": [
    {
      "state": "MA",
      "city": "São Luís",
      "district": "Cohama"
    }
  ],
  "occurrences": [
    {
      "date": "2026-03-15",
      "bodyPart": "Arm",
      "treatment": "Larval removal",
      "outcome": "Cured",
    },
    {
      "date": "2026-03-18",
      "bodyPart": "Ear",
      "treatment": "Antibiotic",
      "outcome": "Under treatment",
    }
  ]
}

Rules:
- Return ONLY the JSON object — no explanations, no markdown fences, no extra text.
- Every field must be present. Use null for any value that cannot be found in the text.
- Do not infer or hallucinate values; only extract information explicitly stated in the source text.
- Dates must follow ISO 8601 format (YYYY-MM-DD).
- Deduplicate hosts, vectors, and locations — reuse existing IDs when the same entity appears in multiple occurrences.
- All ID references in occurrences (hostId, vectorId, locationId) must match an entry in their respective arrays.
- Every Fly name or species matches a vector, not a host"""
                    )}]
    },
    {
        "role": "user",
        "content": [{"type": "text", "text": texto}],
    }
        ]

In [5]:
def run_inference(model, tokenizer, messages):
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False   # desativa o modo de raciocínio interno
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.inference_mode():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=2048,  # suficiente para o JSON completo
            do_sample=False,
        )

    generated_ids = generated_ids[:, inputs["input_ids"].shape[1]:]
    output = tokenizer.decode(generated_ids[0], skip_special_tokens=True)

    # Garante que tokens de thinking residuais não contaminem o JSON
    output = re.sub(r"<think>.*?</think>", "", output, flags=re.DOTALL).strip()

    del inputs, generated_ids
    gc.collect()
    torch.cuda.empty_cache()

    return output

In [6]:
def run_experiment(
    df,
    model_name,
    output_dir="outputs"
):
    not_processed = [13,21]
    os.makedirs(output_dir, exist_ok=True)
    output_path = os.path.join(
        output_dir,
        model_name.replace("/", "_") + "_Sumarization_validation2.jsonl"
    )

    # ==========================
    # Ler ids já processados
    # ==========================
    processed_ids = set()

    if os.path.exists(output_path):
        with open(output_path, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    record = json.loads(line)
                    processed_ids.add(record["id"])
                except:
                    pass

    print(f"{len(processed_ids)} exemplos já processados.")

    model, tokenizer = load_model(model_name)

    with open(output_path, "a", encoding="utf-8") as f:

        for idx, row in df.iterrows():

            if idx in not_processed:
                continue
            # ==========================
            # Pular se já foi respondido
            # ==========================
            if idx in processed_ids:
                print(f"[SKIP] {model_name} | {idx}")
                continue

            texto_extraido = row["sumarization"]

            messages = build_messages(
                texto=texto_extraido
            )

            sumarization = run_inference(
                model=model,
                tokenizer=tokenizer,
                messages=messages
)

            record = {
                "id": idx,
                "model": model_name,
                "sumarization": sumarization
            }

            f.write(json.dumps(record, ensure_ascii=False) + "\n")
            f.flush()
            os.fsync(f.fileno())

            print(f"[{model_name} | {idx}] {sumarization}")

            gc.collect()
            torch.cuda.empty_cache()

    return output_path

In [7]:
run_experiment(
        df=df,
        model_name="Qwen/Qwen3.5-9B"
    )

0 exemplos já processados.


Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 17154.62it/s]
[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights: 100%|██████████| 427/427 [00:01<00:00, 351.24it/s]
[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


[Qwen/Qwen3.5-9B | 0] ,


[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


[Qwen/Qwen3.5-9B | 1] ,


[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


[Qwen/Qwen3.5-9B | 2] {
  "hosts": [
    {
      "sex": "M",
      "age": 60,
      "species": "Homo sapiens",
      "type": "Human"
    }
  ],
  "vectors": [
    {
      "species": "Cochliomyia hominivorax",
      "myiasis_type": "Obrigatória",
      "synonyms": ["blowfly"]
    }
  ],
  "locations": [
    {
      "state": "CE",
      "city": "Camaragibe",
      "district": null
    }
  ],
  "occurrences": [
    {
      "date": "2012-07-01",
      "bodyPart": "Face",
      "treatment": "Ivermectina 12mg via oral em dose única; Debridamento e lavagem com soro fisiológico 0,9%; Remoção de aproximadamente 55 larvas; Antibiótico Cefalexina 500mg de 06 em 06 horas por 07 dias",
      "outcome": "Cured"
    }
  ]
}


KeyboardInterrupt: 